In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.\
config('spark.driver.host','localhost').appName('manipulate').getOrCreate()

## 문자열 다루기, timestamp type으로 변경하기

In [3]:
youtube = spark.read.csv('file:///Users/User/Downloads/youtubevideos.csv',\
                        inferSchema=True,header=True)
youtube.printSchema()

root
 |-- video_id: string (nullable = true)
 |-- trending_date: string (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- publish_time: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: string (nullable = true)
 |-- likes: string (nullable = true)
 |-- dislikes: string (nullable = true)
 |-- comment_count: string (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- comments_disabled: string (nullable = true)
 |-- ratings_disabled: string (nullable = true)
 |-- video_error_or_removed: string (nullable = true)
 |-- description: string (nullable = true)



In [4]:
from pyspark.sql.functions import *

In [5]:
# publish_time의 T, Z 삭제
# 특정 문자 대체 -> regexp_replace
# publish_time의 T와 Z를 삭제한 결과를 publish_time2 열을 생성하여 저장

# T는 공백으로 대체
youtube_tmp = youtube.withColumn('publish_time2',regexp_replace(youtube.publish_time,'T',' ')) 
# Z는 삭제
youtube2 = youtube_tmp.withColumn('publish_time2',regexp_replace(youtube_tmp.publish_time2,'Z',''))

In [6]:
youtube3 = youtube.withColumn('publish_time2',regexp_replace(youtube.publish_time,'T',' ')).\
withColumn('publish_time2',regexp_replace('publish_time2','Z',''))
# Z를 삭제할 때, T가 공백으로 대체된 값이 저장된 publich_time2열을 적용해야 함

In [7]:
print(youtube.collect()[0]['publish_time'])
print(youtube_tmp.collect()[0]['publish_time2'])
print(youtube2.collect()[0]['publish_time2'])
print(youtube3.collect()[0]['publish_time2'])

2017-11-13T17:13:01.000Z
2017-11-13 17:13:01.000Z
2017-11-13 17:13:01.000
2017-11-13 17:13:01.000


In [8]:
# publish_time3 열을 생성하여 publish_time2 값을 date time(timestamp) 타입으로 변환하여 저장
# to_timestamp(column명, 년월일시분초가 저장된 형태)
youtube4 = youtube2.withColumn('publish_time3',\
                    to_timestamp(youtube2.publish_time2,'yyyy-MM-dd HH:mm:ss.SSS'))
# to_timestamp의 결과는 yyyy-MM-dd HH:mm:ss 로 반환됨

In [9]:
youtube4.select('publish_time','publish_time2','publish_time3').show(3)
youtube4.printSchema()

+--------------------+--------------------+-------------------+
|        publish_time|       publish_time2|      publish_time3|
+--------------------+--------------------+-------------------+
|2017-11-13T17:13:...|2017-11-13 17:13:...|2017-11-13 17:13:01|
|2017-11-13T07:30:...|2017-11-13 07:30:...|2017-11-13 07:30:00|
|2017-11-12T19:05:...|2017-11-12 19:05:...|2017-11-12 19:05:24|
+--------------------+--------------------+-------------------+
only showing top 3 rows

root
 |-- video_id: string (nullable = true)
 |-- trending_date: string (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- publish_time: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: string (nullable = true)
 |-- likes: string (nullable = true)
 |-- dislikes: string (nullable = true)
 |-- comment_count: string (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- comments_disabled: str

In [16]:
# translate(column명,'abc','ABC')
# -> a는 A로, b는 B로, c는 C로 대체됨
youtube5 = youtube.withColumn('publish_time4',translate('publish_time','TZ',' '))\
.select('publish_time','publish_time4')
# 'TZ', ' ': T는 공백으로 대체되고, Z는 삭제

# ex> a는 삭제, b는 B로, c는 C로 변경하고자 하는 경우,
#    -> translate('column','abc','BC') -> a는 B로, b는 C로, c는 삭제됨
#    -> translate('column','bca','BC') -> b는 B로, c는 C로, a는 삭제됨

## 새로운 열을 생성하여 새로운 값 할당하기

In [17]:
# likes가 dislikes보다 많으면 good, 나머지 경우 bad 값을 갖는 favor열 추가
# when otherwise 사용
tmp = youtube.withColumn('favor',when(youtube.likes>youtube.dislikes,'good').otherwise('bad'))
tmp.select('likes','dislikes','favor').show(10)

+------+--------+-----+
| likes|dislikes|favor|
+------+--------+-----+
| 57527|    2966| good|
| 97185|    6146| good|
|146033|    5339|  bad|
| 10172|     666|  bad|
|132235|    1989|  bad|
|  9763|     511| good|
| 15993|    2445|  bad|
| 23663|     778|  bad|
|  3543|     119| good|
| 12654|    1363|  bad|
+------+--------+-----+
only showing top 10 rows



In [18]:
# select와 expr 사용
# sql의 CASE WHEN
tmp = youtube.select('likes','dislikes',\
                     expr('CASE WHEN likes>dislikes THEN "good" ELSE "bad" END AS favor'))
tmp.show(10)

+------+--------+-----+
| likes|dislikes|favor|
+------+--------+-----+
| 57527|    2966| good|
| 97185|    6146| good|
|146033|    5339|  bad|
| 10172|     666|  bad|
|132235|    1989|  bad|
|  9763|     511| good|
| 15993|    2445|  bad|
| 23663|     778|  bad|
|  3543|     119| good|
| 12654|    1363|  bad|
+------+--------+-----+
only showing top 10 rows



In [19]:
# trending_date를 date type으로 변경 (to_timestamp와 유사)
# -> yyyy-MM-dd 형태로 반환
# select에 to_date만 사용하는 경우, 처리 결과 값을 저장하는 column이름을 지정할 수 없음
# alias는 처리 결과 값을 저장하는 column 이름을 지정함
youtube.select('trending_date',to_date('trending_date','yy.dd.MM').alias('trending_date2')).show(10)

+-------------+--------------+
|trending_date|trending_date2|
+-------------+--------------+
|     17.14.11|    2017-11-14|
|     17.14.11|    2017-11-14|
|     17.14.11|    2017-11-14|
|     17.14.11|    2017-11-14|
|     17.14.11|    2017-11-14|
|     17.14.11|    2017-11-14|
|     17.14.11|    2017-11-14|
|     17.14.11|    2017-11-14|
|     17.14.11|    2017-11-14|
|     17.14.11|    2017-11-14|
+-------------+--------------+
only showing top 10 rows



In [22]:
youtube.printSchema()

root
 |-- video_id: string (nullable = true)
 |-- trending_date: string (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- publish_time: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: string (nullable = true)
 |-- likes: string (nullable = true)
 |-- dislikes: string (nullable = true)
 |-- comment_count: string (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- comments_disabled: string (nullable = true)
 |-- ratings_disabled: string (nullable = true)
 |-- video_error_or_removed: string (nullable = true)
 |-- description: string (nullable = true)



In [23]:
youtube.select('publish_time').show(10)

+--------------------+
|        publish_time|
+--------------------+
|2017-11-13T17:13:...|
|2017-11-13T07:30:...|
|2017-11-12T19:05:...|
|2017-11-13T11:00:...|
|2017-11-12T18:01:...|
|2017-11-13T19:07:...|
|2017-11-12T05:37:...|
|2017-11-12T21:50:...|
|2017-11-13T14:00:...|
|2017-11-13T13:45:...|
+--------------------+
only showing top 10 rows



In [25]:
youtube = youtube.withColumn('publish_time',translate('publish_time','TZ',' '))

In [28]:
youtube = youtube.withColumn('publish_time',\
                             to_timestamp(youtube.publish_time,'yyyy-MM-dd HH:mm:ss.SSS'))

In [30]:
youtube.printSchema()

root
 |-- video_id: string (nullable = true)
 |-- trending_date: string (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- publish_time: timestamp (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: string (nullable = true)
 |-- likes: string (nullable = true)
 |-- dislikes: string (nullable = true)
 |-- comment_count: string (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- comments_disabled: string (nullable = true)
 |-- ratings_disabled: string (nullable = true)
 |-- video_error_or_removed: string (nullable = true)
 |-- description: string (nullable = true)



In [32]:
youtube = youtube.withColumn('trending_date',to_date(youtube.trending_date,'yy.mm.dd'))

In [33]:
youtube.printSchema()

root
 |-- video_id: string (nullable = true)
 |-- trending_date: date (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- publish_time: timestamp (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: string (nullable = true)
 |-- likes: string (nullable = true)
 |-- dislikes: string (nullable = true)
 |-- comment_count: string (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- comments_disabled: string (nullable = true)
 |-- ratings_disabled: string (nullable = true)
 |-- video_error_or_removed: string (nullable = true)
 |-- description: string (nullable = true)



In [37]:
# datediff 2 date type 또는 timestamp type의 date의 차이를 구함
youtube.select('trending_date','publish_time',datediff('publish_time','trending_date')).show(5)

+-------------+-------------------+-------------------------------------+
|trending_date|       publish_time|datediff(publish_time, trending_date)|
+-------------+-------------------+-------------------------------------+
|   2017-01-11|2017-11-13 17:13:01|                                  306|
|   2017-01-11|2017-11-13 07:30:00|                                  306|
|   2017-01-11|2017-11-12 19:05:24|                                  305|
|   2017-01-11|2017-11-13 11:00:04|                                  306|
|   2017-01-11|2017-11-12 18:01:41|                                  305|
+-------------+-------------------+-------------------------------------+
only showing top 5 rows



In [64]:
youtube.select('publish_time',year(youtube.publish_time).alias('publish_year'),\
               month(youtube.publish_time).alias('publish_month')).show(5)

+-------------------+------------+-------------+
|       publish_time|publish_year|publish_month|
+-------------------+------------+-------------+
|2017-11-13 17:13:01|        2017|           11|
|2017-11-13 07:30:00|        2017|           11|
|2017-11-12 19:05:24|        2017|           11|
|2017-11-13 11:00:04|        2017|           11|
|2017-11-12 18:01:41|        2017|           11|
+-------------------+------------+-------------+
only showing top 5 rows



### string을 다루는 functions

- **ltrim(col)**: string 앞 공백 제거
- **rtrim(col)**: string 뒤 공백 제거
- **trim(col)**: string 앞뒤 공백 제거
- **upper(col)**: string을 대문자로 변경
- **lower(col)**: string을 소문자로 변경
- **initcap(col)**: string의 첫 문자만 대문자, 나머지는 소문자로 변경
- **substring(col,index,length)**: col이 포함한 string의 index부터 length만큼 문자 삭제
- **len(col)**: string의 길이
- **lpad(col,length,str)**: string을 length만큼 확장하고, 좌측을 str로 채움
  ex> lpad('title',10,'a'): big data -> aabig data
- **rpad(col,length,str)**: string을 length만큼 확장하고, 우측을 str로 채움
- **concat(col1,col2)**: col1과 col2의 value를 결합
- **concat_ws(sep,col1,col2)**: col1과 col2를 sep으로 구분하여 결합
- **array_contains(col,val)**: col의 값이 array-like(list, tuple 등) 일 때, 해당 값들 중 val를 포함할 경우 True
- **array_distinct(col)**: col의 값이 array-like(list, tuple 등) 일 때, 중복된 값 제거
- **array_remove(col,val)**: col의 값이 col의 값이 array-like(list, tuple 등) 일 때, val를 포함할 경우, 해당 값에서 val 삭제
- **split(col,val)**: col의 값을 str기준으로 분리

In [38]:
youtube.printSchema()

root
 |-- video_id: string (nullable = true)
 |-- trending_date: date (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- publish_time: timestamp (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: string (nullable = true)
 |-- likes: string (nullable = true)
 |-- dislikes: string (nullable = true)
 |-- comment_count: string (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- comments_disabled: string (nullable = true)
 |-- ratings_disabled: string (nullable = true)
 |-- video_error_or_removed: string (nullable = true)
 |-- description: string (nullable = true)



In [42]:
youtube.select('title',lower(youtube.title)).show(5)

+--------------------+--------------------+
|               title|        lower(title)|
+--------------------+--------------------+
|WE WANT TO TALK A...|we want to talk a...|
|The Trump Preside...|the trump preside...|
|Racist Superman |...|racist superman |...|
|Nickelback Lyrics...|nickelback lyrics...|
|I Dare You: GOING...|i dare you: going...|
+--------------------+--------------------+
only showing top 5 rows



In [53]:
youtube.select('title','channel_title',\
               concat_ws(', [channel]: ',youtube.title,youtube.channel_title).alias('title_and_channel')).show(5)

+--------------------+--------------------+--------------------+
|               title|       channel_title|   title_and_channel|
+--------------------+--------------------+--------------------+
|WE WANT TO TALK A...|        CaseyNeistat|WE WANT TO TALK A...|
|The Trump Preside...|     LastWeekTonight|The Trump Preside...|
|Racist Superman |...|        Rudy Mancuso|Racist Superman |...|
|Nickelback Lyrics...|Good Mythical Mor...|Nickelback Lyrics...|
|I Dare You: GOING...|            nigahiga|I Dare You: GOING...|
+--------------------+--------------------+--------------------+
only showing top 5 rows



In [59]:
title_col = youtube.select('title',split(youtube.title,' ').alias('separated_title'))
# split은 분리된 string을 포함하는 list로 반환함
title_col.show(5)

+--------------------+--------------------+
|               title|     separated_title|
+--------------------+--------------------+
|WE WANT TO TALK A...|[WE, WANT, TO, TA...|
|The Trump Preside...|[The, Trump, Pres...|
|Racist Superman |...|[Racist, Superman...|
|Nickelback Lyrics...|[Nickelback, Lyri...|
|I Dare You: GOING...|[I, Dare, You:, G...|
+--------------------+--------------------+
only showing top 5 rows



In [60]:
title_col.select('title',array_contains(title_col.separated_title,'TO')).show(5)

+--------------------+-----------------------------------+
|               title|array_contains(separated_title, TO)|
+--------------------+-----------------------------------+
|WE WANT TO TALK A...|                               true|
|The Trump Preside...|                              false|
|Racist Superman |...|                              false|
|Nickelback Lyrics...|                              false|
|I Dare You: GOING...|                              false|
+--------------------+-----------------------------------+
only showing top 5 rows



In [61]:
title_col.select('title',array_remove(title_col.separated_title,'TO')).show(5)

+--------------------+---------------------------------+
|               title|array_remove(separated_title, TO)|
+--------------------+---------------------------------+
|WE WANT TO TALK A...|             [WE, WANT, TALK, ...|
|The Trump Preside...|             [The, Trump, Pres...|
|Racist Superman |...|             [Racist, Superman...|
|Nickelback Lyrics...|             [Nickelback, Lyri...|
|I Dare You: GOING...|             [I, Dare, You:, G...|
+--------------------+---------------------------------+
only showing top 5 rows

